# Train DMFGCRN SUMO on Google Colab

Notebook này chạy training SUMO với cấu hình hiện tại:

- Input: `data/SumoData/SUMO.npz`
- Target: `flow_count_5m`, `e1_occupancy`, `speed`
- Input-only mask: `is_missing_speed`
- Forecast horizon: `48`
- Summary horizons: `1, 3, 6, 12, 24, 36, 48`

Trước khi chạy, upload toàn bộ thư mục project `DMFGCRN` lên Google Drive, ví dụ: `/content/drive/MyDrive/DMFGCRN`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Sửa path này nếu bạn đặt project ở chỗ khác trên Drive.
PROJECT_DIR = '/content/drive/MyDrive/DMFGCRN'
%cd $PROJECT_DIR

In [ ]:
!python --version
!nvidia-smi
!pip install -q pandas matplotlib tqdm openpyxl

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Verify Or Build SUMO Data

Nếu đã có `data/SumoData/SUMO.npz`, cell này chỉ kiểm tra shape. Nếu chưa có nhưng còn CSV gốc, nó sẽ chạy script process.

In [ ]:
from pathlib import Path
import numpy as np

sumo_npz = Path('data/SumoData/SUMO.npz')
raw_csv = Path('data/SumoData/forecast_baseline_node_level.csv')

if not sumo_npz.exists():
    assert raw_csv.exists(), 'Missing both SUMO.npz and forecast_baseline_node_level.csv'
    !python data/SumoData/process_sumo_data.py

z = np.load(sumo_npz)
data = z['data']
print('SUMO data shape:', data.shape)
print('feature_names:', z['feature_names'].tolist() if 'feature_names' in z.files else 'not saved')
print('finite:', np.isfinite(data).all())
print('is_missing_speed count:', int(data[..., 3].sum()))
assert data.shape[1:] == (30, 4)
assert np.isfinite(data).all()

## Configure Training

Colab thường chạy nhanh hơn local, nhưng `horizon=48` vẫn tốn thời gian. Có thể tăng `EPOCHS` sau khi smoke train ổn.

In [ ]:
from pathlib import Path
import configparser

DATASET = 'SUMO'
MODEL = 'DMFGCRN'
HORIZON = 48
BATCH_SIZE = 192
EPOCHS = 50
SEED = 10

config_path = Path(f'config_file/{DATASET}_{MODEL}.conf')
config = configparser.ConfigParser()
config.read(config_path)

config['data']['num_nodes'] = '30'
config['data']['lag'] = '12'
config['data']['horizon'] = str(HORIZON)
config['data']['steps_per_day'] = '288'
config['model']['input_dim'] = '4'
config['model']['output_dim'] = '3'
config['train']['batch_size'] = str(BATCH_SIZE)
config['train']['epochs'] = str(EPOCHS)
config['train']['seed'] = str(SEED)

with config_path.open('w') as f:
    config.write(f)

# Ensure run.py uses one seed only. This avoids running 5 full trainings in Colab.
run_py = Path('run.py')
run_text = run_py.read_text()
if 'seeds = [args.seed]' not in run_text:
    run_text = run_text.replace('print(args)\n\n# init_seed', 'print(args)\n\nseeds = [args.seed]\n\n# init_seed')
    run_py.write_text(run_text)

print(config_path.read_text())

## Smoke Test Shapes

Cell này chỉ chạy 1 forward pass để bắt lỗi shape trước khi train lâu.

In [ ]:
import argparse
import torch
import torch.nn as nn
from lib.dataloader import get_dataloader
from model.DMFGCRN import DMFGCRN

args = argparse.Namespace(
    dataset='SUMO', val_ratio=0.2, test_ratio=0.2, lag=12, horizon=HORIZON,
    num_nodes=30, steps_per_day=288, input_dim=4, output_dim=3,
    normalizer='std', column_wise=False, batch_size=min(BATCH_SIZE, 16),
    rnn_units=64, num_layers=1, cheb_k=2, embed_dim=8,
    use_day=True, use_week=True, default_graph=True,
)
train_loader, val_loader, test_loader, scaler = get_dataloader(
    args, normalizer=args.normalizer, tod=False, dow=False, weather=False, single=False
)
x, y = next(iter(train_loader))
model = DMFGCRN(args).to(x.device)
for p in model.parameters():
    if p.dim() > 1:
        nn.init.xavier_uniform_(p)
    else:
        nn.init.uniform_(p)
out = model(x)
print('x:', tuple(x.shape), x.device)
print('y:', tuple(y.shape))
print('out:', tuple(out.shape))
print('label:', tuple(y[..., :args.output_dim].shape))
print('finite:', bool(torch.isfinite(out).all()))
assert out.shape == y[..., :args.output_dim].shape
assert torch.isfinite(out).all()

## Train

In [ ]:
!python -u run.py --dataset SUMO --model DMFGCRN --device cuda:0 | tee train_sumo_colab.log

## Metric Summary For Selected Horizons

In [ ]:
from pathlib import Path
import pandas as pd

selected_horizons = [1, 3, 6, 12, 24, 36, 48]
exp_root = Path('experiments/SUMO')
metric_files = sorted(exp_root.glob('*/10/horizon_metrics.csv'), key=lambda p: p.stat().st_mtime)
if not metric_files:
    metric_files = sorted(exp_root.glob('**/horizon_metrics.csv'), key=lambda p: p.stat().st_mtime)
assert metric_files, 'No horizon_metrics.csv found. Training may not have reached final test.'

metrics_path = metric_files[-1]
df = pd.read_csv(metrics_path)
df['horizon_str'] = df['horizon'].astype(str)
summary = df[df['horizon_str'].isin([str(h) for h in selected_horizons] + ['average'])]
summary = summary.drop(columns=['horizon_str'])
print('metrics file:', metrics_path)
display(summary)

summary_path = metrics_path.with_name('selected_horizon_metrics.csv')
summary.to_csv(summary_path, index=False)
print('saved:', summary_path)

## Download Outputs

In [ ]:
from google.colab import files

# Uncomment nếu muốn download file summary về máy.
# files.download(str(summary_path))
# files.download('train_sumo_colab.log')